# 03 — Evaluation & Error Analysis (v1)

Giai đoạn 4 (`docs/PLAN.md`) — spec: `docs/specs/g4-evaluation-error-analysis.md`.

Mục tiêu: confusion matrix trên model baseline (config **ON** đã chốt ở
Giai đoạn 3 — `runs/detect/yolov8n_v1_baseline`), xác định cặp lớp hay bị
nhầm nhất, trực quan hoá bằng **EigenCAM** (không dùng Grad-CAM chuẩn —
không áp dụng thẳng được cho detection, xem `docs/REQUIREMENTS.md` mục 7),
và tóm tắt pattern lỗi làm input cho Giai đoạn 5.

**Chạy trên Colab** (cần GPU — Runtime → Change runtime type → T4 GPU).

## Setup — mount Drive + cd vào repo

Giống hệt 2 notebook trước. Thêm assert kiểm tra `best.pt` của baseline đã
có sẵn (train ở Giai đoạn 2/3, không train lại ở đây).

In [ ]:
import os

REPO_DIR_NAME = "computer-vision-project"  # đổi nếu bạn git clone ra tên thư mục khác

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_path = f"/content/drive/MyDrive/{REPO_DIR_NAME}"
    if not os.path.isdir(drive_path):
        raise FileNotFoundError(
            f"{drive_path} không tồn tại — kiểm tra lại bạn đã `git clone` repo vào "
            "đúng chỗ trong Drive chưa (T0.4), hoặc sửa REPO_DIR_NAME ở trên cho khớp."
        )
    os.chdir(drive_path)
except ImportError:
    pass  # không chạy trên Colab (vd Jupyter local) — giả định cwd đã là repo root

print("cwd:", os.getcwd())
assert os.path.isdir("scripts") and os.path.isdir("data"), (
    "Chưa đứng ở repo root — không thấy scripts/ và data/ ở cwd hiện tại."
)
BEST_WEIGHTS = "runs/detect/yolov8n_v1_baseline/weights/best.pt"
assert os.path.isfile(BEST_WEIGHTS), (
    f"{BEST_WEIGHTS} chưa có — chạy notebooks/02_train_detector.ipynb "
    "(Giai đoạn 2/3) trước để có checkpoint baseline."
)

## Đồng bộ code mới nhất

In [ ]:
assert os.path.isdir(".git"), (
    "cwd hiện tại không phải repo root (không thấy .git) — runtime Colab có "
    "thể vừa bị reset. Chạy lại cell 'Setup — mount Drive + cd vào repo' ở "
    "trên (mount + cd) trước, rồi chạy lại cell này."
)
!git pull

## Cài dependencies

In [ ]:
!pip install -q -r requirements.txt
import torch
import ultralytics
from pytorch_grad_cam import EigenCAM  # noqa: F401 — sanity-check import ngay

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## T4.1–T4.2 — Confusion matrix + cặp lớp hay nhầm nhất

`load_confusion_matrix()`/`top_confused_pairs()` (`src/evaluation/metrics.py`)
chạy `model.val()` 1 lần trên `best.pt`, lấy ma trận thật, xếp hạng cặp
lớp bị nhầm nhiều nhất (bỏ qua `background` — chỉ quan tâm nhầm lẫn *giữa
các tư thế thật* với nhau).

In [ ]:
from src.evaluation.metrics import load_confusion_matrix, top_confused_pairs

DATA_YAML = "data/raw/yoga_v1/data.yaml"

matrix, class_names_cm = load_confusion_matrix(BEST_WEIGHTS, DATA_YAML)
class_names = [c for c in class_names_cm if c != "background"]
confused_pairs = top_confused_pairs(matrix, class_names_cm, k=3)

print("Confusion matrix shape:", matrix.shape)
print("Top cặp lớp hay nhầm nhất (thật -> dự đoán: số lần):")
if confused_pairs:
    for true_c, pred_c, count in confused_pairs:
        print(f"  {true_c} -> {pred_c}: {count}")
else:
    print("  (không có cặp nào bị nhầm — model gần như hoàn hảo trên test set)")

## T4.3 — EigenCAM: chọn target layer

In cấu trúc `model.model.model` (backbone → neck → `Detect` head) để chọn
đúng layer cuối backbone/neck — **không đoán mù index**, nhìn tên layer
thật rồi mới chốt `TARGET_LAYER_INDEX` ở cell dưới.

In [ ]:
from ultralytics import YOLO

yolo = YOLO(BEST_WEIGHTS)
torch_model = yolo.model  # nn.Module PyTorch bên trong (DetectionModel)
torch_model.eval()

for i, layer in enumerate(torch_model.model):
    print(i, layer.__class__.__name__)

Từ danh sách trên, chọn layer **ngay trước `Detect`** (phần tử cuối cùng) —
mặc định `TARGET_LAYER_INDEX = -2`. Nếu heatmap ở cell sanity-check dưới
trông vô nghĩa (sáng đều toàn ảnh, hoặc chỉ tập trung 1 góc không liên
quan), thử đổi sang `-3`/`-4` và chạy lại.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

TARGET_LAYER_INDEX = -2
target_layers = [torch_model.model[TARGET_LAYER_INDEX]]
cam = EigenCAM(torch_model, target_layers)


def preprocess_image(image_path: Path, imgsz: int = 640) -> tuple[torch.Tensor, np.ndarray]:
    """BGR file -> (input tensor NCHW float32 0-1, RGB float32 0-1 HWC for overlay)."""
    img_bgr = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (imgsz, imgsz))
    rgb_float = np.float32(img_resized) / 255.0
    tensor = torch.from_numpy(rgb_float).permute(2, 0, 1).unsqueeze(0)
    return tensor, rgb_float


def eigencam_overlay(image_path: Path) -> np.ndarray:
    tensor, rgb_float = preprocess_image(image_path)
    # targets=[] (khác None cố ý): nếu targets=None, BaseCAM.forward() cố tự
    # suy ra target từ output bằng outputs.cpu().data.numpy() — nhưng output
    # thô của model detection là 1 tuple, không phải tensor đơn, nên crash
    # (AttributeError: 'tuple' object has no attribute 'cpu'). EigenCAM
    # (uses_gradients=False, get_cam_image() chỉ dùng activations) không hề
    # đọc targets, nên truyền list rỗng là an toàn — chỉ cần khác None để
    # né nhánh suy luận target đó.
    grayscale_cam = cam(tensor, targets=[])[0, :, :]
    return show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True)


# Sanity-check trên 1 ảnh mẫu bất kỳ từ test set.
TEST_IMAGES_DIR = Path("data/raw/yoga_v1/test/images")
TEST_LABELS_DIR = Path("data/raw/yoga_v1/test/labels")
sample_path = next(TEST_IMAGES_DIR.glob("*.*"))

plt.imshow(eigencam_overlay(sample_path))
plt.axis("off")
plt.title(f"EigenCAM sanity-check: {sample_path.name}")
plt.show()

## T4.4 — 3-5 ảnh EigenCAM: đúng vs nhầm

Chạy `predict()` trên toàn bộ test set (nhỏ, ~101 ảnh — đủ nhanh), so với
nhãn thật để tách 2 nhóm: dự đoán sai (misclassified) và dự đoán đúng
(correct). Ưu tiên minh hoạ đúng cặp lớp hay nhầm nhất ở T4.2 (nếu có),
rồi bù thêm ảnh dự đoán đúng ở các lớp khác nhau cho đủ 3-5 ảnh.

In [ ]:
def true_class_of(image_path: Path) -> str | None:
    label_path = TEST_LABELS_DIR / (image_path.stem + ".txt")
    if not label_path.exists():
        return None
    lines = [l for l in label_path.read_text().splitlines() if l.strip()]
    if not lines:
        return None
    return class_names[int(lines[0].split()[0])]


def predict_top_class(image_path: Path) -> tuple[str | None, float]:
    result = yolo.predict(source=str(image_path), verbose=False)[0]
    if len(result.boxes) == 0:
        return None, 0.0
    top_idx = int(result.boxes.conf.argmax())
    class_id = int(result.boxes.cls[top_idx])
    conf = float(result.boxes.conf[top_idx])
    return class_names[class_id], conf


test_images = sorted(TEST_IMAGES_DIR.glob("*.jpg")) + sorted(TEST_IMAGES_DIR.glob("*.png"))
classified = []
for img_path in test_images:
    true_cls = true_class_of(img_path)
    if true_cls is None:
        continue
    pred_cls, conf = predict_top_class(img_path)
    classified.append({"path": img_path, "true": true_cls, "pred": pred_cls, "conf": conf})

misclassified = [c for c in classified if c["pred"] != c["true"]]
correct = [c for c in classified if c["pred"] == c["true"]]
print(f"{len(misclassified)} ảnh dự đoán sai / {len(correct)} ảnh dự đoán đúng (trên {len(classified)} ảnh test)")

samples = []
if confused_pairs:
    top_true, top_pred, _ = confused_pairs[0]
    match = next((c for c in misclassified if c["true"] == top_true and c["pred"] == top_pred), None)
    if match:
        samples.append(match)
samples += misclassified[: max(0, 2 - len(samples))]

seen_classes = {s["true"] for s in samples}
for c in correct:
    if len(samples) >= 5:
        break
    if c["true"] not in seen_classes:
        samples.append(c)
        seen_classes.add(c["true"])

print(f"Chọn {len(samples)} ảnh để minh hoạ EigenCAM.")

In [ ]:
fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
if len(samples) == 1:
    axes = [axes]
for ax, s in zip(axes, samples):
    ax.imshow(eigencam_overlay(s["path"]))
    ax.axis("off")
    status = "ĐÚNG" if s["pred"] == s["true"] else "SAI"
    ax.set_title(f"{status}: thật={s['true']}, đoán={s['pred']} ({s['conf']:.2f})", fontsize=9)
plt.tight_layout()
plt.show()

**Nhận xét mỗi ảnh (điền sau khi chạy):** _với mỗi ảnh ở trên — model nhìn
đúng vùng học viên hay bị phân tâm bởi nền/vật thể khác? Có pattern nào
khác nhau giữa ảnh dự đoán đúng và ảnh dự đoán sai không?_

## T4.5 — Tóm tắt pattern lỗi (điền sau khi chạy)

_Từ T4.2 (cặp lớp hay nhầm) + T4.4 (quan sát EigenCAM): lớp nào hay nhầm
với lớp nào, và giả thuyết vì sao (dáng giống nhau, vật nhỏ/bị che khuất,
ánh sáng, nền phân tâm...). Đủ cụ thể để dùng trực tiếp cho T5.1 (chọn
hướng cải thiện ở Giai đoạn 5)._